In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from sklearn.utils.class_weight import compute_class_weight


In [ ]:
# ─────────────────────────────────────────────
# 1. Carga
# ─────────────────────────────────────────────
df = pd.read_csv('data_set_unbalanced.csv', sep=';')
print(f"Dataset cargado: {df.shape}")


In [ ]:
df.head()

In [ ]:
# ─────────────────────────────────────────────
# 2. Descartar columna índice
# ─────────────────────────────────────────────
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
# ─────────────────────────────────────────────
# 3. Decodificar edad (formato SINAN brasileño)
#    4XXX = años  |  3XXX = días  |  2XXX = meses  |  1XXX = años
# ─────────────────────────────────────────────
def decode_idade(val):
    if val >= 4000:   return val - 4000
    elif val >= 3000: return (val - 3000) / 365
    elif val >= 2000: return (val - 2000) / 12
    elif val >= 1000: return val - 1000
    else:             return val / 365

df['IDADE_ANOS'] = df['NU_IDADE_N'].apply(decode_idade)
df['IDADE_ANOS'] = df['IDADE_ANOS'].clip(upper=110)   # cap outliers (15 registros > 110 años)
df = df.drop(columns=['NU_IDADE_N'])

In [ ]:
# ─────────────────────────────────────────────
# 4. Parsear DIAS → número entero de días
#    Cap en p95 (~113 días) para eliminar errores de fecha
# ─────────────────────────────────────────────
df['DIAS_NUM'] = df['DIAS'].str.extract(r'(\d+)').astype(float)
cap_dias = df['DIAS_NUM'].quantile(0.95)
df['DIAS_NUM'] = df['DIAS_NUM'].clip(upper=cap_dias)
df = df.drop(columns=['DIAS'])
print(f"DIAS capado en {cap_dias:.0f} días (p95)")

In [ ]:
# ─────────────────────────────────────────────
# 5. Recodificar síntomas: 1→1 (sí), 2→0 (no)
#    En SINAN: 1 = presente, 2 = ausente
# ─────────────────────────────────────────────
symptom_cols = [
    'FEBRE','MIALGIA','CEFALEIA','EXANTEMA','VOMITO','NAUSEA',
    'DOR_COSTAS','CONJUNTVIT','ARTRITE','ARTRALGIA','PETEQUIA_N',
    'LEUCOPENIA','LACO','DOR_RETRO','DIABETES','HEMATOLOG',
    'HEPATOPAT','RENAL','HIPERTENSA','ACIDO_PEPT','AUTO_IMUNE'
]
for col in symptom_cols:
    df[col] = (df[col] == 1).astype(int)

In [ ]:
df.head()

In [ ]:
# ─────────────────────────────────────────────
# 6. One-hot encoding en variables categóricas
#    El valor 9 (ignorado/desconocido) se conserva como categoría
# ─────────────────────────────────────────────
cat_cols = ['CS_GESTANT', 'CS_RACA', 'CS_ZONA', 'CS_SEXO']
df[cat_cols] = df[cat_cols].astype(int).astype(str)
df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)

In [ ]:
# ─────────────────────────────────────────────
# REEMPLAZA el bloque 7 del preprocesamiento
# Target binario: 1 = CHIKUNGUNYA, 0 = todo lo demás
# ─────────────────────────────────────────────
y = (df['CLASSI_FIN'] == 'CHIKUNGUNYA').astype(int).values

print("Distribución target binario:")
print(f"  CHIKUNGUNYA (1): {y.sum():>6}  ({y.mean()*100:.1f}%)")
print(f"  OTHER+DENGUE (0): {(y==0).sum():>6}  ({(1-y.mean())*100:.1f}%)")

In [ ]:
# ─────────────────────────────────────────────
# 7. Codificar target
#    CHIKUNGUNYA=0 | DENGUE=1 | OTHER=2
# ─────────────────────────────────────────────
le = LabelEncoder()
df['LABEL'] = le.fit_transform(df['CLASSI_FIN'])
class_names = le.classes_
print(f"Clases: {dict(zip(class_names, le.transform(class_names)))}")

X = df.drop(columns=['CLASSI_FIN', 'LABEL'])
y = df['LABEL'].values
print(f"Features finales: {X.shape[1]} columnas")

In [ ]:
X.head()

In [ ]:
y

In [ ]:
# ─────────────────────────────────────────────
# 8. Split estratificado 70 / 15 / 15
# ─────────────────────────────────────────────
RANDOM_STATE = 42

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)
print(f"\nSplit  →  Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}")


In [ ]:
# ─────────────────────────────────────────────
# 9. Escalar solo IDADE_ANOS y DIAS_NUM
#    Fit únicamente en train
# ─────────────────────────────────────────────
scale_cols = ['IDADE_ANOS', 'DIAS_NUM']
scale_idx  = [list(X.columns).index(c) for c in scale_cols]

scaler = StandardScaler()
X_train_arr = X_train.values.astype(np.float32)
X_val_arr   = X_val.values.astype(np.float32)
X_test_arr  = X_test.values.astype(np.float32)

X_train_arr[:, scale_idx] = scaler.fit_transform(X_train_arr[:, scale_idx])
X_val_arr  [:, scale_idx] = scaler.transform(X_val_arr[:, scale_idx])
X_test_arr [:, scale_idx] = scaler.transform(X_test_arr[:, scale_idx])

In [ ]:
# ─────────────────────────────────────────────
# 10. Class weights con boost ×1.5 en CHIKUNGUNYA
# ─────────────────────────────────────────────
raw_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {i: w for i, w in enumerate(raw_weights)}
print("Class weights ajustados:")
for i, name in enumerate(class_names):
    print(f"  {name:>15}: {class_weight_dict[i]:.4f}")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

RANDOM_STATE = 42
N_FEATURES = X_train_arr.shape[1]  # 44
N_CLASSES  = 3
tf.random.set_seed(RANDOM_STATE)

In [ ]:
# ─────────────────────────────────────────────
# Arquitectura
# Input(44) → Dense(128, ReLU) → BN → Dropout(0.3)
#           → Dense(64,  ReLU) → BN → Dropout(0.3)
#           → Dense(32,  ReLU) →      Dropout(0.2)
#           → Dense(3, Softmax)
# ─────────────────────────────────────────────
def build_model(n_features, n_classes):
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),

        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),

        layers.Dense(n_classes, activation='softmax')
    ])
    return model

model = build_model(N_FEATURES, N_CLASSES)
model.summary()

In [ ]:
# ─────────────────────────────────────────────
# Compilar
# ─────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# ─────────────────────────────────────────────
# Callbacks
# ─────────────────────────────────────────────
cb_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1
    )
]

In [ ]:
# ─────────────────────────────────────────────
# Entrenamiento
# ─────────────────────────────────────────────
history = model.fit(
    X_train_arr, y_train,
    validation_data=(X_val_arr, y_val),
    epochs=100,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=cb_list,
    verbose=1
)

print("\n✓ Entrenamiento finalizado")
print(f"  Épocas entrenadas: {len(history.history['loss'])}")
print(f"  Mejor val_loss:    {min(history.history['val_loss']):.4f}")
print(f"  Mejor val_accuracy:{max(history.history['val_accuracy']):.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Predicciones en validación
y_pred_proba = model.predict(X_val_arr)
y_pred = np.argmax(y_pred_proba, axis=1)

# Reporte por clase
print(classification_report(y_val, y_pred, target_names=class_names))

# Matriz de confusión
print("Matriz de confusión:")
print(confusion_matrix(y_val, y_pred))

In [ ]:
import pandas as pd
import numpy as np

# Comparar prevalencia de síntomas por clase en el dataset original
symptom_cols = [
    'FEBRE','MIALGIA','CEFALEIA','EXANTEMA','VOMITO','NAUSEA',
    'DOR_COSTAS','CONJUNTVIT','ARTRITE','ARTRALGIA','PETEQUIA_N',
    'LEUCOPENIA','LACO','DOR_RETRO','DIABETES','HEMATOLOG',
    'HEPATOPAT','RENAL','HIPERTENSA','ACIDO_PEPT','AUTO_IMUNE'
]

# Si no lo guardaste, usa esto:
feature_names = list(X.columns)  # X del paso 1

df_analysis = pd.DataFrame(X_train_arr, columns=feature_names)
df_analysis['clase'] = y_train

# Media de cada síntoma por clase (recuerda: 1=presente, 0=ausente)
symptom_idx = [feature_names.index(c) for c in symptom_cols if c in feature_names]
symptom_names = [feature_names[i] for i in symptom_idx]

result = df_analysis.groupby('clase')[symptom_names].mean().T
result.columns = class_names
result['diferencia_max'] = result.max(axis=1) - result.min(axis=1)
result = result.sort_values('diferencia_max', ascending=False)

print("Prevalencia de síntomas por clase (ordenado por diferencia):")
print(result.round(3).to_string())

Ahora el diagnóstico es muy claro. El dataset tiene un problema estructural:
Solo 2 síntomas realmente diferencian las clases:

ARTRALGIA — CHIKUNGUNYA 0.83 vs DENGUE 0.36 vs OTHER 0.22 → diferencia 0.61 ✓
DOR_RETRO — diferencia 0.15, el resto ya es ruido

Todo lo demás tiene diferencias menores a 0.14. FEBRE por ejemplo está presente en las tres clases con prevalencia 0.91/0.92/0.81 — no aporta nada discriminativo.
Esto explica perfectamente por qué la red no puede superar 58% — la información simplemente no está en el dataset para separar las tres clases con confianza.
Tienes tres opciones realistas:
1. Reducir a clasificación binaria     → CHIKUNGUNYA vs NO-CHIKUNGUNYA
   (más alcanzable, útil clínicamente)

2. Quedarte con features discriminativas → solo ARTRALGIA + top 5 síntomas
   + edad + días + demográficos

3. Aceptar el límite del dataset y      → reportar resultados honestos
   enfocarte en maximizar recall        con las métricas correctas
   de CHIKUNGUNYA
Dado que el objetivo del proyecto es detectar CHIKUNGUNYA, la opción 1 es la más poderosa. Conviertes el problema a binario y la red puede concentrarse en aprender exactamente esa frontera.